# ENEMDU 2021-2025 — Pipeline Medallón (Bronze → Silver → Gold)

**Proyecto:** UCUENCA-SABE · MVP Dashboard de Empleabilidad
**Fuente:** Encuesta Nacional de Empleo, Desempleo y Subempleo (ENEMDU) — INEC, microdatos anuales de personas.

## Objetivo del caso de uso
1. Analizar la **empleabilidad de graduados universitarios** (2021-2025).
2. Identificar **sobrecalificación**: graduados ocupados en ocupaciones que no requieren título superior (CIUO-08, grandes grupos 4-9).
3. Generar **agregados listos para Power BI**.

## Estructura del notebook
| Sección | Contenido | Módulo |
|---|---|---|
| 1 | Ingesta Bronze (Kaggle) | `src/ingestion/kaggle_downloader.py` |
| 2 | **Objetivo 1** — Auditoría y análisis de esquemas | `enemdu_schema_analyzer.py` |
| 3 | **Objetivo 2** — Construcción de Silver | `enemdu_silver_builder.py` |
| 4 | **Objetivo 3** — Métricas de empleabilidad (Gold) | `enemdu_gold_builder.py` |
| 5 | **Objetivo 4** — Validación, visualizaciones y export | notebook |

> **Nota:** la capa Bronze está reducida a modo de ejemplo; el pipeline está diseñado para el volumen completo (lectura por `usecols`, tipado explícito y Parquet comprimido).

---
## 1. Ingesta — Capa Bronze
Descarga desde Kaggle y clasificación en `diccionarios/` y `microdatos_csv/`. Bronze es **inmutable**: sirve de respaldo y trazabilidad; ninguna transformación se escribe aquí.

Si ya se descargó ya no correr la kaggle sino toma 9min aprox

In [1]:
import sys
from pathlib import Path

# Inyectar la raíz del proyecto al sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CONFIG, get_external_bronze_dir
from src.ingestion.kaggle_downloader import download_and_organize_enemdu

print(f"✅ Proyecto cargado correctamente: {CONFIG['project']['name']}")
print(f"📁 Ruta raíz: {PROJECT_ROOT}")
print(f"📊 Sources disponibles: {list(CONFIG['sources'].keys())}")

✅ Proyecto cargado correctamente: ucuenca-sabe
📁 Ruta raíz: c:\Users\michu\mis-proyectos\ucuenca-sabe
📊 Sources disponibles: ['externas', 'internas', 'investigacion', 'enemdu']


In [2]:
# Ahora descarga
print("🚀 Iniciando descarga desde Kaggle...")
try:
    inventario = download_and_organize_enemdu()
    print("✅ ¡Proceso completado!")
except Exception as e:
    print(f"❌ Error: {e}")

🚀 Iniciando descarga desde Kaggle...
🚀 Autenticando con Kaggle API...
✅ Autenticación exitosa con Access Token.
📥 Descargando dataset 'kmichelle/enemdu-ecuador-microdatos-anuales-personas' en: C:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu
Dataset URL: https://www.kaggle.com/datasets/kmichelle/enemdu-ecuador-microdatos-anuales-personas
✅ Descarga finalizada.
❌ Error en download_and_organize_enemdu: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\michu\\mis-proyectos\\ucuenca-sabe\\data\\bronze\\externas\\enemdu\\BDDenemdu_personas_2021_anual.csv' -> 'C:\\Users\\michu\\mis-proyectos\\ucuenca-sabe\\data\\bronze\\externas\\enemdu\\microdatos_csv\\BDDenemdu_personas_2021_anual.csv'
❌ Error: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\michu\\mis-proyectos\\ucuenca-sabe\\data\\bronze\\externas\\enemdu\\BDDenemdu_personas_2021_anual.csv' -> 'C:\\Users\\michu\\mis-proyectos\\ucuenca-sabe\\data\\bronze\\externas\\enemdu\\m

### 1.1 Chequeo de años nuevos
Kaggle no expone una API de "cambios"; la estrategia es comparar los años presentes en local contra el año esperado y volver a descargar sólo si falta alguno.

In [3]:
from pathlib import Path

def check_nuevos_anios(anio_esperado: int = 2025, descargar: bool = False) -> list[int]:
    """Devuelve los años ausentes en Bronze y, opcionalmente, relanza la ingesta."""
    csv_dir = PROJECT_ROOT / "data" / "bronze" / "externas" / "enemdu" / "microdatos_csv"
    anios_locales = sorted(
        int(m.group()) for f in csv_dir.glob("*.csv")
        if (m := __import__("re").search(r"(?:19|20)\d{2}", f.stem))
    )
    faltantes = [a for a in range(2021, anio_esperado + 1) if a not in anios_locales]

    print(f"📂 Años locales: {anios_locales}")
    if faltantes:
        print(f"📥 Faltan años: {faltantes}")
        if descargar:
            download_and_organize_enemdu()
    else:
        print("✅ Ya tienes los datos más recientes")
    return faltantes

check_nuevos_anios(2025)

📂 Años locales: [2021, 2022, 2023, 2024, 2025]
✅ Ya tienes los datos más recientes


[]

In [4]:
print(CONFIG["sources"].keys())

dict_keys(['externas', 'internas', 'investigacion', 'enemdu'])


---
## 2. Objetivo 1 — Auditoría y análisis de esquemas

**Pregunta que responde esta sección:** ¿qué contiene realmente Bronze y qué se puede llevar a Silver sin romper la comparabilidad 2021-2025?

`ENEMDUSchemaAnalyzer` hace cuatro cosas:
1. Lee **todos los diccionarios XLSX** (detectando automáticamente la fila de encabezado, que cambia entre años).
2. Lee **sólo la cabecera** de cada CSV (detectando encoding, separador y decimal — el INEC alterna `;`/`,` y UTF-8/Latin-1).
3. Cruza *esquema declarado* (diccionario) vs. *esquema real* (CSV).
4. Audita calidad: nulos, duplicados de llave, rangos y presencia de variables críticas.

In [5]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src.processing import ENEMDUSchemaAnalyzer, ENEMDUSilverBuilder, ENEMDUGoldBuilder
from src.processing.io_utils import ENEMDUPaths

paths = ENEMDUPaths.build(PROJECT_ROOT).ensure()
for k, v in paths.as_dict().items():
    print(f"{k:>14}: {v}")

          root: c:\Users\michu\mis-proyectos\ucuenca-sabe
        bronze: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu
  diccionarios: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu\diccionarios
    microdatos: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu\microdatos_csv
        silver: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\silver\enemdu
          gold: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu
       reports: c:\Users\michu\mis-proyectos\ucuenca-sabe\reports


### 2.1 Inventario de la capa Bronze

In [6]:
analyzer = ENEMDUSchemaAnalyzer(paths=paths, years=range(2021, 2026))
inventario_bronze = analyzer.inventory()
display(inventario_bronze)

print(f"\nTotal archivos: {len(inventario_bronze)} | "
      f"Peso Bronze: {inventario_bronze['peso_mb'].sum():.1f} MB")

09:30:14 | INFO    | src.processing.enemdu_schema_analyzer | Bronze ENEMDU: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\bronze\externas\enemdu


,tipo,archivo,anio,peso_mb,ruta
0,microdato,BDDenemdu_personas_2021_anual.csv,2021,155.36,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
1,microdato,BDDenemdu_personas_2022_anual.csv,2022,145.27,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
2,microdato,BDDenemdu_personas_2023_anual.csv,2023,141.42,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
3,microdato,BDDenemdu_personas_2024_anual.csv,2024,138.55,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
4,microdato,BDDenemdu_personas_2025_anual.csv,2025,135.88,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
5,diccionario,Diccionario de Datos_persona_anual_2021.xlsx,2021,0.02,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
6,diccionario,Diccionario de Datos_persona_anual_2023.xlsx,2023,0.02,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
7,diccionario,Diccionario de Datos_persona_anual_2024.xlsx,2024,0.02,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
8,diccionario,Diccionario de Datos_persona_anual_2025.xlsx,2025,0.02,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...
9,diccionario,Diccionario de Datos_personas_anual_2022.xlsx,2022,0.01,c:\Users\michu\mis-proyectos\ucuenca-sabe\data...



Total archivos: 10 | Peso Bronze: 716.6 MB


### 2.2 Análisis de esquemas por año

In [7]:
schemas = analyzer.analyze_all()
resumen = analyzer.summary()
display(resumen)

print("Lectura:")
print("  · cobertura_% = columnas del CSV documentadas en el diccionario")
print("  · solo_en_dicc = variables documentadas que NO llegaron al CSV (revisar entrega del INEC)")
print("  · cols_exclusivas = columnas que existen únicamente en ese año")

09:30:16 | INFO    | src.processing.enemdu_schema_analyzer | Año 2021 → 151 cols CSV | 153 cols diccionario | cobertura 100.0%
09:30:16 | INFO    | src.processing.enemdu_schema_analyzer | Año 2022 → 139 cols CSV | 141 cols diccionario | cobertura 100.0%
09:30:17 | INFO    | src.processing.enemdu_schema_analyzer | Año 2023 → 141 cols CSV | 143 cols diccionario | cobertura 100.0%
09:30:18 | INFO    | src.processing.enemdu_schema_analyzer | Año 2024 → 139 cols CSV | 141 cols diccionario | cobertura 100.0%
09:30:19 | INFO    | src.processing.enemdu_schema_analyzer | Año 2025 → 139 cols CSV | 141 cols diccionario | cobertura 100.0%


,anio,csv,diccionario,filas,cols_csv,cols_dicc,cobertura_%,solo_en_csv,solo_en_dicc,cols_exclusivas
0,2021,BDDenemdu_personas_2021_anual.csv,Diccionario de Datos_persona_anual_2021.xlsx,361790,151,153,100.0,0,2,12
1,2022,BDDenemdu_personas_2022_anual.csv,Diccionario de Datos_personas_anual_2022.xlsx,358096,139,141,100.0,0,2,0
2,2023,BDDenemdu_personas_2023_anual.csv,Diccionario de Datos_persona_anual_2023.xlsx,345174,141,143,100.0,0,2,2
3,2024,BDDenemdu_personas_2024_anual.csv,Diccionario de Datos_persona_anual_2024.xlsx,341394,139,141,100.0,0,2,0
4,2025,BDDenemdu_personas_2025_anual.csv,Diccionario de Datos_persona_anual_2025.xlsx,334786,139,141,100.0,0,2,0


Lectura:
  · cobertura_% = columnas del CSV documentadas en el diccionario
  · solo_en_dicc = variables documentadas que NO llegaron al CSV (revisar entrega del INEC)
  · cols_exclusivas = columnas que existen únicamente en ese año


### 2.3 Columnas comunes vs. específicas por año

In [8]:
comunes = analyzer.common_columns
especificas = analyzer.year_specific

print(f"🔗 Columnas comunes a los {len(schemas)} años: {len(comunes)}")
print(f"🌐 Unión de columnas (todos los años): {len(analyzer.all_columns)}\n")

for anio, cols in especificas.items():
    print(f"  {anio}: {len(cols)} exclusivas → {cols[:12]}{' ...' if len(cols) > 12 else ''}")

matriz = analyzer.presence_matrix()
print("\nDistribución de columnas por estado:")
display(matriz["estado"].value_counts().to_frame("n_columnas"))
display(matriz.head(25))

🔗 Columnas comunes a los 5 años: 139
🌐 Unión de columnas (todos los años): 153

  2021: 12 exclusivas → ['p59', 'p60a', 'p60b', 'p60c', 'p60d', 'p60e', 'p60f', 'p60g', 'p60h', 'p60i', 'p60j', 'p60k']
  2022: 0 exclusivas → []
  2023: 2 exclusivas → ['p081', 'p085']
  2024: 0 exclusivas → []
  2025: 0 exclusivas → []

Distribución de columnas por estado:


,n_columnas
estado,
común,139
exclusiva,14


,2021,2022,2023,2024,2025,n_anios,estado
area,True,True,True,True,True,5,común
ced01a,True,True,True,True,True,5,común
ciudad,True,True,True,True,True,5,común
cod_inf,True,True,True,True,True,5,común
condact,True,True,True,True,True,5,común
conglomerado,True,True,True,True,True,5,común
desempleo,True,True,True,True,True,5,común
dominio,True,True,True,True,True,5,común
empleo,True,True,True,True,True,5,común
epobreza,True,True,True,True,True,5,común


### 2.4 Resolución de variables críticas
El INEC renombra variables entre publicaciones (`p51a` → `totalhoras`, `ingrl` → `ingreso_laboral`, etc.).
Aquí se resuelve, **año por año**, qué nombre real corresponde a cada variable del modelo de Silver.
Una fila `⚠️ parcial` o `❌ ausente` es una decisión de diseño pendiente, no un error silencioso.

In [9]:
criticas = analyzer.resolve_critical()
display(criticas)

pendientes = criticas[criticas["estado"] != "✅ completa"]
if len(pendientes):
    print("⚠️ Variables a revisar manualmente en el diccionario:")
    display(pendientes[["variable_silver", "estado"]])
else:
    print("✅ Todas las variables críticas se resolvieron en los 5 años.")

,variable_silver,2021,2022,2023,2024,2025,disponible_en,estado
0,anio,periodo,periodo,periodo,periodo,periodo,5,✅ completa
1,mes,mes,mes,mes,mes,mes,5,✅ completa
2,provincia,prov,prov,prov,prov,prov,5,✅ completa
3,ciudad,ciudad,ciudad,ciudad,ciudad,ciudad,5,✅ completa
4,area,area,area,area,area,area,5,✅ completa
5,sexo,p02,p02,p02,p02,p02,5,✅ completa
6,edad,p03,p03,p03,p03,p03,5,✅ completa
7,nivel_instruccion,p10a,p10a,p10a,p10a,p10a,5,✅ completa
8,anios_aprobados,p10b,p10b,p10b,p10b,p10b,5,✅ completa
9,condicion_actividad,condact,condact,condact,condact,condact,5,✅ completa


✅ Todas las variables críticas se resolvieron en los 5 años.


### 2.5 Etiquetas de valores (código → significado)
Los mapeos **no se asumen**: se extraen del diccionario del año. Sólo si no son interpretables se aplica el mapeo *fallback* documentado en `enemdu_mappings.py`, y la procedencia queda registrada en el reporte de calidad.

In [10]:
for var in ["p10a", "condact", "p02", "area", "rama1"]:
    labels = analyzer.labels_for(var)
    print(f"\n▸ {var} ({len(labels)} categorías)")
    for cod, txt in list(labels.items())[:12]:
        print(f"    {cod:>3} = {txt}")
    if not labels:
        print("    (sin etiquetas en el diccionario → se usará el mapeo fallback)")


▸ p10a (0 categorías)
    (sin etiquetas en el diccionario → se usará el mapeo fallback)

▸ condact (0 categorías)
    (sin etiquetas en el diccionario → se usará el mapeo fallback)

▸ p02 (0 categorías)
    (sin etiquetas en el diccionario → se usará el mapeo fallback)

▸ area (0 categorías)
    (sin etiquetas en el diccionario → se usará el mapeo fallback)

▸ rama1 (0 categorías)
    (sin etiquetas en el diccionario → se usará el mapeo fallback)


### 2.6 Auditoría de calidad de Bronze

In [11]:
auditoria = analyzer.audit(sample_rows=100_000)
display(auditoria)

ruta_reporte = analyzer.save_report()
print(f"📄 Reporte de esquemas: {ruta_reporte}")

,anio,filas_totales,filas_muestra,columnas,peso_mb,encoding,sep,pct_nulos_medio,cols_100pct_nulas,llave_detectada,dups_llave,edad_min,edad_max,fexp_presente,criticas_faltantes
0,2021,361790,100000,151,155.36,utf-8,;,0.0,0,id_persona + id_hogar + conglomerado + viviend...,0,0.0,98.0,True,—
1,2022,358096,100000,139,145.27,utf-8,;,0.0,0,id_persona + id_hogar + conglomerado + viviend...,0,0.0,98.0,True,—
2,2023,345174,100000,141,141.42,utf-8,;,0.0,0,id_persona + id_hogar + conglomerado + viviend...,0,0.0,98.0,True,—
3,2024,341394,100000,139,138.55,utf-8,;,0.0,0,id_persona + id_hogar + conglomerado + viviend...,0,0.0,98.0,True,—
4,2025,334786,100000,139,135.88,utf-8,;,0.0,0,id_persona + id_hogar + conglomerado + viviend...,0,0.0,98.0,True,—


09:30:38 | INFO    | src.processing.enemdu_schema_analyzer | Reporte de esquemas guardado en c:\Users\michu\mis-proyectos\ucuenca-sabe\reports\schemas\schema_analysis_20260904.json


📄 Reporte de esquemas: c:\Users\michu\mis-proyectos\ucuenca-sabe\reports\schemas\schema_analysis_20260904.json


### 2.7 Decisiones de diseño para Silver

A partir de la auditoría se fija el contrato de la capa Silver:

| Decisión | Criterio |
|---|---|
| **Selección de columnas** | No se copia el CSV completo: sólo las variables del modelo de empleabilidad (identificación, geografía, demografía, educación, mercado laboral, ponderador). |
| **Esquema estable** | La unión de variables canónicas se mantiene en todos los años; lo que no exista en un año se completa con `NULL` (NaN), nunca con ceros. |
| **Nombres legibles** | `p02` → `sexo_cod` → `sexo`; `condact` → `condicion_actividad`. Se conserva el código original (`*_cod`) junto a la etiqueta, para trazabilidad y para Power BI. |
| **Códigos de no respuesta** | `999999`, `99`, `-1` en variables continuas → `NaN` (no se imputa en Silver). |
| **Rangos válidos** | edad 0-110, horas 0-126, ingreso 0-100 000, `fexp` ≥ 0. Fuera de rango → `NaN` + registro en el reporte. |
| **Deduplicación** | Sólo si existe identificador de persona real; de lo contrario se advierte y no se borra nada. |
| **Sin imputación ni filtros analíticos** | Filtrar PET (≥15 años) o graduados es responsabilidad de **Gold**, no de Silver. |
| **Formato** | Parquet + Snappy, un archivo unificado (`enemdu_unificado.parquet`). |

---
## 3. Objetivo 2 — Construcción de la capa Silver

`ENEMDUSilverBuilder` aplica, por año: resolución de alias → lectura selectiva → renombrado → esquema estable → tipado y limpieza → decodificación semántica → variables derivadas; y finalmente concatena 2021-2025.

**Derivadas generadas:** `es_pea`, `es_ocupado`, `es_desempleado`, `es_subempleado`, `es_empleo_adecuado`, `es_graduado_superior`, `es_posgrado`, `ciuo_gran_grupo` (+ descripción y nivel de competencia), `ocupacion_requiere_superior`, **`es_sobrecalificado`**, `grupo_edad`, `ingreso_por_hora`, `provincia`.

In [12]:
# SAMPLE = 50_000 para iterar rápido; None = procesar todo (producción)
SAMPLE = None

silver_builder = ENEMDUSilverBuilder(analyzer=analyzer, paths=paths)
resultado = silver_builder.build(sample_rows=SAMPLE)
silver = resultado.df

print(f"✅ Silver: {silver.shape[0]:,} filas × {silver.shape[1]} columnas")
display(silver.head())

09:30:38 | INFO    | src.processing.enemdu_silver_builder | Año 2021 → 27/28 variables canónicas resueltas
09:30:58 | INFO    | src.processing.enemdu_silver_builder | Año 2022 → 27/28 variables canónicas resueltas
09:31:14 | INFO    | src.processing.enemdu_silver_builder | Año 2023 → 27/28 variables canónicas resueltas
09:31:31 | INFO    | src.processing.enemdu_silver_builder | Año 2024 → 27/28 variables canónicas resueltas
09:31:46 | INFO    | src.processing.enemdu_silver_builder | Año 2025 → 27/28 variables canónicas resueltas
09:32:04 | INFO    | src.processing.enemdu_silver_builder | Deduplicación por ['anio', 'mes', 'conglomerado', 'vivienda', 'hogar', 'id_persona']: -1215640 filas


✅ Silver: 525,600 filas × 50 columnas


,anio,mes,provincia,area,sexo,edad,grupo_edad,nivel_instruccion,es_graduado_superior,condicion_actividad,es_ocupado,es_desempleado,es_sobrecalificado,ciuo_gran_grupo_desc,rama_actividad,ingreso_laboral,horas_trabajadas,factor_expansion,area_cod,codigo_ciudad,conglomerado,vivienda,hogar,parentesco_cod,sexo_cod,estado_civil_cod,afiliacion_cod,asiste_clases_cod,nivel_instruccion_cod,anios_aprobados,ocupacion_cod,rama_cod,categoria_ocupacion_cod,condicion_actividad_cod,sector_empleo_cod,grupo_ocupacion_cod,rama_agrupada_cod,codigo_provincia,id_persona,empleo_informal_cod,fuente,ciuo_gran_grupo,ciuo_nivel_competencia,es_subempleado,es_empleo_adecuado,es_pea,es_edad_trabajar,es_posgrado,ocupacion_requiere_superior,ingreso_por_hora
0,2021,7,Azuay,Urbana,Hombre,58.0,55-64,Superior universitaria,True,Empleo no clasificado,True,False,True,"Oficiales, operarios y artesanos",C. Industrias manufactureras,NaN,40.0,7.918958,1,10150,802,2,1,1,1,1,2,2,9,3.0,7522,6,2,6,1,7,3,1,1.015000e+20,<NA>,ENEMDU 2021 (INEC),7,2,False,False,True,True,False,False,NaN
4,2021,1,Azuay,Urbana,Mujer,84.0,65+,Secundaria,False,Población económicamente inactiva (PEI),False,False,False,NaN,NaN,NaN,NaN,7.428837,1,10150,2101,2,1,1,2,1,10,2,6,6.0,<NA>,<NA>,<NA>,9,<NA>,<NA>,<NA>,1,1.015000e+20,<NA>,ENEMDU 2021 (INEC),<NA>,<NA>,False,False,False,True,False,False,NaN
6,2021,1,Azuay,Urbana,Hombre,48.0,45-54,Superior universitaria,True,Desempleo abierto,False,True,False,Trabajadores de servicios y vendedores,NaN,NaN,NaN,7.428837,1,10150,2101,7,1,1,1,1,10,2,9,4.0,5223,2,<NA>,7,<NA>,<NA>,<NA>,1,1.015000e+20,<NA>,ENEMDU 2021 (INEC),5,2,False,False,True,True,False,False,NaN
9,2021,6,Azuay,Urbana,Hombre,59.0,55-64,Superior universitaria,True,Empleo no clasificado,True,False,False,Técnicos y profesionales de nivel medio,Q. Salud humana y asistencia social,NaN,40.0,8.310475,1,10150,3003,5,1,1,1,1,1,2,9,3.0,3255,6,2,6,1,3,17,1,1.015000e+20,<NA>,ENEMDU 2021 (INEC),3,3,False,False,True,True,False,True,NaN
13,2021,6,Azuay,Urbana,Mujer,53.0,45-54,Secundaria,False,Empleo no clasificado,True,False,False,Personal de apoyo administrativo,O. Administración pública y defensa,NaN,40.0,8.310475,1,10150,3003,9,1,1,2,1,1,2,6,6.0,4226,1,<NA>,6,1,4,15,1,1.015000e+20,<NA>,ENEMDU 2021 (INEC),4,2,False,False,True,True,False,False,NaN


### 3.1 Mapeo aplicado por año (trazabilidad)

In [13]:
display(resultado.mapping_report.set_index("anio").T)

print("Procedencia de los mapeos de códigos (diccionario | fallback):")
display(pd.DataFrame(resultado.provenance).T)

anio,periodo,periodo,periodo,periodo,periodo
mes,mes,mes,mes,mes,mes
conglomerado,conglomerado,conglomerado,conglomerado,conglomerado,conglomerado
vivienda,vivienda,vivienda,vivienda,vivienda,vivienda
hogar,hogar,hogar,hogar,hogar,hogar
id_persona,id_persona,id_persona,id_persona,id_persona,id_persona
factor_expansion,fexp,fexp,fexp,fexp,fexp
codigo_ciudad,ciudad,ciudad,ciudad,ciudad,ciudad
codigo_provincia,prov,prov,prov,prov,prov
area_cod,area,area,area,area,area
sexo_cod,p02,p02,p02,p02,p02


Procedencia de los mapeos de códigos (diccionario | fallback):


,sexo,area,nivel_instruccion,condicion_actividad,rama_actividad
2021,fallback,fallback,fallback,fallback,fallback
2022,fallback,fallback,fallback,fallback,fallback
2023,fallback,fallback,fallback,fallback,fallback
2024,fallback,fallback,fallback,fallback,fallback
2025,fallback,fallback,fallback,fallback,fallback


### 3.2 Validación de consistencia y calidad

In [14]:
display(resultado.quality)

# Consistencia del tamaño muestral y la población expandida entre años
variacion = resultado.quality.set_index("anio")[["filas", "poblacion_expandida"]].pct_change() * 100
print("Variación interanual (%):")
display(variacion.round(2))

,anio,filas,poblacion_expandida,edad_media,%_nulos_nivel_instruccion,%_nulos_condicion_actividad,%_nulos_ciuo,%_nulos_ingreso,graduados_superior,ocupados,desempleados,mapeo_educacion,mapeo_condact
0,2021,103488,4720373.0,52.3,0.0,0.00,21.39,27.38,20859,78598,2839,fallback,fallback
1,2022,106362,4894465.0,53.1,0.0,0.00,22.00,26.53,21352,80784,2274,fallback,fallback
2,2023,105237,5041726.0,53.4,0.0,0.01,23.07,27.27,21073,79004,2046,fallback,fallback
3,2024,105775,5152352.0,54.2,0.0,0.00,24.14,28.03,21123,78395,1946,fallback,fallback
4,2025,104738,5183385.0,54.2,0.0,0.00,24.67,28.24,21532,77370,1654,fallback,fallback


Variación interanual (%):


,filas,poblacion_expandida
anio,,
2021,NaN,NaN
2022,2.78,3.69
2023,-1.06,3.01
2024,0.51,2.19
2025,-0.98,0.60


In [15]:
# Nulos por columna (top 20) y tipos resultantes
nulos = (silver.isna().mean() * 100).sort_values(ascending=False).round(2)
display(nulos.head(20).to_frame("%_nulos"))

print("\nDistribución de la condición de actividad:")
display(silver["condicion_actividad"].value_counts(dropna=False, normalize=True).mul(100).round(2).to_frame("%"))

print("Distribución del nivel de instrucción:")
display(silver["nivel_instruccion"].value_counts(dropna=False, normalize=True).mul(100).round(2).to_frame("%"))

,%_nulos
empleo_informal_cod,100.00
categoria_ocupacion_cod,61.61
ingreso_por_hora,27.49
ingreso_laboral,27.49
sector_empleo_cod,25.01
grupo_ocupacion_cod,25.01
rama_agrupada_cod,25.01
rama_actividad,25.01
horas_trabajadas,25.01
ciuo_nivel_competencia,23.06



Distribución de la condición de actividad:


,%
condicion_actividad,
Empleo adecuado/pleno,34.72
Otro empleo no pleno,24.02
Población económicamente inactiva (PEI),22.96
Subempleo por insuficiencia de tiempo de trabajo,13.34
Desempleo abierto,1.75
Subempleo por insuficiencia de ingresos,1.69
Empleo no remunerado,0.89
Empleo no clasificado,0.33
Desempleo oculto,0.30


Distribución del nivel de instrucción:


,%
nivel_instruccion,
Primaria,36.24
Secundaria,32.66
Superior universitaria,16.77
Ninguno,3.71
Post-grado,3.39
Educación Media/Bachillerato,3.11
Superior no universitaria,2.92
Educación Básica,0.98
Centro de alfabetización,0.22


In [16]:
# Chequeos de integridad: rangos, coherencia lógica y cobertura del ponderador
checks = {
    "edad fuera de rango": int(((silver["edad"] < 0) | (silver["edad"] > 110)).sum()),
    "ingreso negativo": int((silver["ingreso_laboral"] < 0).sum()),
    "ocupado y desempleado a la vez": int((silver["es_ocupado"] & silver["es_desempleado"]).sum()),
    "sobrecalificado sin ser graduado": int((silver["es_sobrecalificado"] & ~silver["es_graduado_superior"]).sum()),
    "sobrecalificado no ocupado": int((silver["es_sobrecalificado"] & ~silver["es_ocupado"]).sum()),
    "filas sin factor de expansión": int(silver["factor_expansion"].isna().sum()),
    "años presentes": silver["anio"].nunique(),
}
display(pd.Series(checks).to_frame("resultado"))
assert checks["ocupado y desempleado a la vez"] == 0, "Inconsistencia en la condición de actividad"
print("✅ Chequeos de integridad superados")

,resultado
edad fuera de rango,0
ingreso negativo,0
ocupado y desempleado a la vez,0
sobrecalificado sin ser graduado,0
sobrecalificado no ocupado,0
filas sin factor de expansión,0
años presentes,5


✅ Chequeos de integridad superados


### 3.3 Persistencia de Silver

In [17]:
ruta_silver = silver_builder.save(resultado, partition_by_year=False)
print(f"💾 {ruta_silver}")
print(f"   Tamaño: {ruta_silver.stat().st_size / 1e6:.2f} MB")

09:32:09 | INFO    | src.processing.enemdu_silver_builder | Silver guardada: c:\Users\michu\mis-proyectos\ucuenca-sabe\data\silver\enemdu\enemdu_unificado.parquet (525600 filas, 50 cols, 11.4 MB)


💾 c:\Users\michu\mis-proyectos\ucuenca-sabe\data\silver\enemdu\enemdu_unificado.parquet
   Tamaño: 11.41 MB


---
## 4. Objetivo 3 — Métricas de empleabilidad (capa Gold)

### Definiciones operativas
| Concepto | Definición |
|---|---|
| **Graduado superior** | Nivel de instrucción *Superior universitaria* o *Post-grado*, con 24 años o más (evita contar a quienes aún cursan). |
| **PET** | Población de 15 años y más. |
| **PEA** | Ocupados + desempleados. |
| **Tasa de desempleo** | Desempleados / PEA. |
| **Tasa de empleo adecuado** | Empleo adecuado-pleno / PEA. |
| **Sobrecalificación** | Graduado ocupado cuyo gran grupo CIUO-08 es 4-9 (niveles de competencia 1-2, no requieren título universitario). |
| **Ponderación** | Todos los indicadores usan el **factor de expansión** (`fexp`); se reporta `n_muestral` y la marca `confiable` (n ≥ 30). |

In [18]:
gold_builder = ENEMDUGoldBuilder(silver, paths=paths, edad_minima=15, edad_minima_graduado=24)
gold = gold_builder.build()

print("📊 KPI anual — graduados universitarios")
display(gold.kpi_anual[[
    "anio", "n_muestral", "poblacion", "tasa_participacion", "tasa_empleo",
    "tasa_desempleo", "tasa_empleo_adecuado", "tasa_subempleo",
    "tasa_sobrecalificacion", "ingreso_laboral_medio"
]])

09:32:09 | INFO    | src.processing.enemdu_gold_builder | PET (>= 15 años): 525581 de 525600 filas
09:32:09 | INFO    | src.processing.enemdu_gold_builder | Construyendo Gold sobre 525581 filas de Silver...


📊 KPI anual — graduados universitarios


,anio,n_muestral,poblacion,tasa_participacion,tasa_empleo,tasa_desempleo,tasa_empleo_adecuado,tasa_subempleo,tasa_sobrecalificacion,ingreso_laboral_medio
0,2021,20616,717895.0,83.16,96.24,3.76,66.64,14.40,42.91,1028.35
1,2022,21068,706013.0,83.13,97.15,2.85,71.43,11.82,41.90,1034.81
2,2023,20722,714751.0,82.93,97.51,2.49,73.06,9.90,41.93,1032.34
3,2024,20751,708083.0,82.82,97.63,2.37,72.67,11.06,40.69,1051.80
4,2025,21140,704946.0,82.43,97.60,2.40,72.50,10.09,42.24,1028.01


### 4.1 Sobrecalificación por gran grupo ocupacional (CIUO-08)

In [19]:
sobre = gold.sobrecalificacion_ocupacion
display(sobre[sobre["anio"] == sobre["anio"].max()])

resumen_sobre = (sobre.groupby(["anio", "requiere_titulo_superior"])["ocupados"]
                 .sum().unstack().rename(columns={True: "en_ocupación_acorde",
                                                  False: "en_ocupación_no_acorde"}))
resumen_sobre["% sobrecalificación"] = (
    100 * resumen_sobre["en_ocupación_no_acorde"]
    / resumen_sobre.sum(axis=1)).round(2)
display(resumen_sobre)

,anio,ciuo_gran_grupo,ciuo_gran_grupo_desc,requiere_titulo_superior,n_muestral,ocupados,participacion_ocupados_pct,ingreso_laboral_medio
36,2025,1,Directores y gerentes,True,1119,33332.0,5.88,2056.77
37,2025,2,Profesionales científicos e intelectuales,True,6763,227579.0,40.13,1231.13
38,2025,3,Técnicos y profesionales de nivel medio,True,2130,66661.0,11.75,1099.84
39,2025,4,Personal de apoyo administrativo,False,796,30751.0,5.42,820.33
40,2025,5,Trabajadores de servicios y vendedores,False,2587,95816.0,16.89,773.11
41,2025,6,Agricultores y trabajadores calificados agrope...,False,538,22669.0,4.00,388.87
42,2025,7,"Oficiales, operarios y artesanos",False,890,37107.0,6.54,591.73
43,2025,8,Operadores de instalaciones y máquinas,False,674,32516.0,5.73,618.39
44,2025,9,Ocupaciones elementales,False,528,20715.0,3.65,406.08


requiere_titulo_superior,en_ocupación_no_acorde,en_ocupación_acorde,% sobrecalificación
anio,,,
2021,246509.0,328028.0,42.91
2022,238898.0,331280.0,41.90
2023,242363.0,335671.0,41.93
2024,232953.0,339556.0,40.69
2025,239574.0,327572.0,42.24


### 4.2 Territorio, sexo/edad, rama y comparación por nivel educativo

In [20]:
ultimo = int(silver["anio"].max())

print(f"▸ Top 10 provincias por tasa de desempleo de graduados ({ultimo})")
prov = gold.por_provincia.query("anio == @ultimo and confiable")
display(prov.nlargest(10, "tasa_desempleo")[
    ["provincia", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "tasa_sobrecalificacion", "ingreso_laboral_medio"]])

print(f"▸ Brecha de género ({ultimo})")
display(gold.por_sexo_edad.query("anio == @ultimo")[
    ["sexo", "grupo_edad", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "tasa_sobrecalificacion", "ingreso_laboral_medio", "brecha_desempleo_pp", "confiable"]])

print(f"▸ Ramas de actividad que absorben graduados ({ultimo})")
display(gold.por_rama.query("anio == @ultimo").nlargest(10, "participacion_pct"))

print("▸ Comparativo por nivel de instrucción (todos los niveles)")
display(gold.comparativo_nivel.query("anio == @ultimo")[
    ["nivel_instruccion", "n_muestral", "tasa_desempleo", "tasa_empleo_adecuado",
     "ingreso_laboral_medio", "tasa_sobrecalificacion"]])

▸ Top 10 provincias por tasa de desempleo de graduados (2025)


,provincia,n_muestral,tasa_desempleo,tasa_empleo_adecuado,tasa_sobrecalificacion,ingreso_laboral_medio
99,Cañar,222,9.75,69.92,39.12,879.52
103,Esmeraldas,909,7.68,60.16,46.08,776.41
107,Loja,647,5.94,65.39,38.35,901.28
97,Bolívar,340,5.67,69.99,48.19,971.39
104,Galápagos,235,5.40,83.31,41.52,1706.91
112,Orellana,169,4.74,65.27,42.90,1061.19
116,Santo Domingo de los Tsáchilas,208,3.75,79.37,47.90,877.02
113,Pastaza,396,3.63,69.66,38.63,1061.49
106,Imbabura,602,3.34,64.40,47.13,863.33
101,Cotopaxi,367,3.06,52.87,43.47,857.21


▸ Brecha de género (2025)


,sexo,grupo_edad,n_muestral,tasa_desempleo,tasa_empleo_adecuado,tasa_sobrecalificacion,ingreso_laboral_medio,brecha_desempleo_pp,confiable
48,Hombre,15-24,77,6.56,48.29,84.77,711.08,-1.82,True
49,Hombre,25-34,1674,1.32,78.97,49.05,962.44,5.07,True
50,Hombre,35-44,2763,1.80,80.50,38.59,1153.36,1.98,True
51,Hombre,45-54,2732,1.38,79.61,38.00,1243.53,1.28,True
52,Hombre,55-64,2936,2.60,66.00,46.13,1090.59,-1.17,True
53,Hombre,65+,3123,1.37,46.42,54.10,855.08,-1.37,True
54,Mujer,15-24,75,4.74,84.90,40.84,620.21,-1.82,True
55,Mujer,25-34,1209,6.39,67.92,42.47,844.35,5.07,True
56,Mujer,35-44,1795,3.78,69.43,45.66,888.99,1.98,True
57,Mujer,45-54,1749,2.66,74.11,30.46,988.13,1.28,True


▸ Ramas de actividad que absorben graduados (2025)


,anio,rama_actividad,n_muestral,ocupados,participacion_pct,sobrecalificados,tasa_sobrecalificacion,ingreso_laboral_medio
99,2025,P. Enseñanza,2806,98505.0,17.37,2958.0,3.00,1121.50
90,2025,G. Comercio al por mayor y menor,2534,86964.0,15.33,67918.0,78.10,857.89
98,2025,O. Administración pública y defensa,1653,55269.0,9.75,12719.0,23.01,1431.77
86,2025,C. Industrias manufactureras,1310,49964.0,8.81,30204.0,60.45,1098.73
96,2025,"M. Actividades profesionales, científicas y té...",1490,44141.0,7.78,1785.0,4.04,946.20
100,2025,Q. Salud humana y asistencia social,1310,43641.0,7.69,4455.0,10.21,1258.79
84,2025,"A. Agricultura, ganadería, silvicultura y pesca",767,32698.0,5.77,26675.0,81.58,620.67
91,2025,H. Transporte y almacenamiento,700,28943.0,5.10,25574.0,88.36,764.44
92,2025,I. Alojamiento y servicios de comida,613,24274.0,4.28,20914.0,86.16,663.70
89,2025,F. Construcción,540,19340.0,3.41,10914.0,56.43,785.01


▸ Comparativo por nivel de instrucción (todos los niveles)


,nivel_instruccion,n_muestral,tasa_desempleo,tasa_empleo_adecuado,ingreso_laboral_medio,tasa_sobrecalificacion
38,Centro de alfabetización,179,0.00,11.29,172.56,NaN
39,Educación Básica,1304,3.07,22.38,328.76,NaN
40,Educación Media/Bachillerato,4700,3.49,43.02,444.95,NaN
41,Ninguno,3837,0.99,10.12,215.94,NaN
42,Post-grado,4099,1.48,88.48,1595.55,9.77
43,Primaria,36731,1.17,26.99,356.52,NaN
44,Secundaria,33013,1.89,48.09,515.02,NaN
45,Superior no universitaria,3439,2.66,73.52,865.78,NaN
46,Superior universitaria,17433,2.64,68.98,902.55,49.37


### 4.3 Persistencia de Gold

In [21]:
salidas = gold_builder.save(gold, export_csv=True)
for nombre, ruta in salidas.items():
    print(f"  {nombre:<32} → {ruta}")

09:32:11 | INFO    | src.processing.enemdu_gold_builder | Gold guardada: 7 tablas en c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu


  empleabilidad_graduados          → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\empleabilidad_graduados.parquet
  kpi_anual                        → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\kpi_anual.parquet
  empleabilidad_provincia          → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\empleabilidad_provincia.parquet
  empleabilidad_sexo_edad          → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\empleabilidad_sexo_edad.parquet
  sobrecalificacion_ocupacion      → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\sobrecalificacion_ocupacion.parquet
  graduados_rama_actividad         → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\graduados_rama_actividad.parquet
  comparativo_nivel_instruccion    → c:\Users\michu\mis-proyectos\ucuenca-sabe\data\gold\enemdu\comparativo_nivel_instruccion.parquet


---
## 5. Objetivo 4 — Validación y reporte

### 5.1 Tamaños de muestra y precisión

In [22]:
muestra = (silver.assign(grad=silver["es_graduado_superior"])
           .groupby("anio")
           .agg(filas_silver=("anio", "size"),
                pet=("es_edad_trabajar", "sum"),
                graduados=("grad", "sum"),
                pea_graduados=("es_pea", "sum"))
           .assign(pct_graduados=lambda d: (100 * d["graduados"] / d["filas_silver"]).round(2)))
display(muestra)

alerta = gold.por_provincia.query("not confiable")
print(f"⚠️ Celdas provincia×año con n < 30 (no publicables): {len(alerta)}")
if len(alerta):
    display(alerta[["anio", "provincia", "n_muestral"]].head(15))

,filas_silver,pet,graduados,pea_graduados,pct_graduados
anio,,,,,
2021,103488,103486,20859,81437,20.16
2022,106362,106358,21352,83058,20.07
2023,105237,105231,21073,81050,20.02
2024,105775,105771,21123,80341,19.97
2025,104738,104735,21532,79024,20.56


⚠️ Celdas provincia×año con n < 30 (no publicables): 0


### 5.2 Validación contra cifras oficiales (opcional)
Completa el diccionario con la tasa de desempleo **nacional** publicada por el INEC para cada año
(boletín anual de ENEMDU). La comparación se calcula sobre el total de la PEA, no sólo graduados;
diferencias menores a ~0,5 pp son esperables por redondeo y por el submuestreo de Bronze.

In [23]:
# Rellenar con los valores del boletín oficial del INEC (dejar None si no se dispone)
TASAS_OFICIALES_INEC = {2021: None, 2022: None, 2023: None, 2024: None, 2025: None}

nacional = ENEMDUGoldBuilder(silver, paths=paths).\
    _rates(ENEMDUGoldBuilder._aggregate(
        silver[silver["edad"].ge(15).fillna(False)], ["anio"]))[["anio", "tasa_desempleo"]]
nacional = nacional.rename(columns={"tasa_desempleo": "tasa_desempleo_calculada"})
nacional["tasa_desempleo_INEC"] = nacional["anio"].map(TASAS_OFICIALES_INEC)
nacional["diferencia_pp"] = (nacional["tasa_desempleo_calculada"]
                             - nacional["tasa_desempleo_INEC"]).round(2)
display(nacional)

if nacional["tasa_desempleo_INEC"].notna().any():
    peor = nacional["diferencia_pp"].abs().max()
    print(f"Máxima diferencia: {peor:.2f} pp", "✅ dentro de lo esperado" if peor < 1 else "⚠️ revisar definiciones")
else:
    print("ℹ️ Sin cifras oficiales cargadas: completa TASAS_OFICIALES_INEC para activar la validación.")

09:32:12 | INFO    | src.processing.enemdu_gold_builder | PET (>= 15 años): 525581 de 525600 filas


,anio,tasa_desempleo_calculada,tasa_desempleo_INEC,diferencia_pp
0,2021,2.98,None,NaN
1,2022,2.35,None,NaN
2,2023,2.18,None,NaN
3,2024,1.89,None,NaN
4,2025,1.80,None,NaN


ℹ️ Sin cifras oficiales cargadas: completa TASAS_OFICIALES_INEC para activar la validación.


### 5.3 Visualizaciones exploratorias

In [24]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9})

kpi = gold.kpi_anual.sort_values("anio")
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0, 0].plot(kpi["anio"], kpi["tasa_desempleo"], marker="o", color="#FF6B35")
axes[0, 0].set_title("Tasa de desempleo — graduados universitarios")
axes[0, 0].set_ylabel("%")

axes[0, 1].plot(kpi["anio"], kpi["tasa_empleo_adecuado"], marker="o", color="#1B998B")
axes[0, 1].plot(kpi["anio"], kpi["tasa_subempleo"], marker="s", color="#C7522A")
axes[0, 1].legend(["Empleo adecuado", "Subempleo"], frameon=False)
axes[0, 1].set_title("Calidad del empleo de graduados")
axes[0, 1].set_ylabel("% de la PEA graduada")

axes[1, 0].bar(kpi["anio"], kpi["tasa_sobrecalificacion"], color="#2E4057")
axes[1, 0].set_title("Tasa de sobrecalificación (CIUO-08 grupos 4-9)")
axes[1, 0].set_ylabel("% de graduados ocupados")

comp = gold.comparativo_nivel.query("anio == @ultimo").nlargest(8, "ingreso_laboral_medio")
axes[1, 1].barh(comp["nivel_instruccion"], comp["ingreso_laboral_medio"], color="#6C8EAD")
axes[1, 1].set_title(f"Ingreso laboral medio por nivel de instrucción ({ultimo})")
axes[1, 1].set_xlabel("USD/mes")

fig.suptitle("ENEMDU — Empleabilidad de graduados universitarios (Gold)", fontsize=12)
fig.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Brecha de género y dispersión territorial
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sexo = (gold.por_sexo_edad.query("confiable")
        .groupby(["anio", "sexo"])["tasa_desempleo"].mean().unstack())
sexo.plot(marker="o", ax=axes[0], color={"Hombre": "#2E4057", "Mujer": "#FF6B35"})
axes[0].set_title("Desempleo de graduados por sexo")
axes[0].set_ylabel("%")
axes[0].legend(frameon=False)

top = gold.por_provincia.query("anio == @ultimo and confiable").nlargest(12, "tasa_sobrecalificacion")
axes[1].barh(top["provincia"], top["tasa_sobrecalificacion"], color="#1B998B")
axes[1].invert_yaxis()
axes[1].set_title(f"Sobrecalificación por provincia ({ultimo})")
axes[1].set_xlabel("% de graduados ocupados")

fig.tight_layout()
plt.show()

### 5.4 Exportación para Power BI

In [ ]:
export_dir = paths.reports / "powerbi"
archivos = sorted(export_dir.glob("*.csv"))

print(f"📤 {len(archivos)} tablas exportadas en {export_dir}\n")
for f in archivos:
    n = sum(1 for _ in f.open(encoding="utf-8-sig")) - 1
    print(f"  {f.name:<40} {n:>6} filas   {f.stat().st_size/1e3:>7.1f} KB")

print("\nModelo sugerido en Power BI:")
print("  · Tabla de hechos : empleabilidad_graduados (anio × provincia × área × sexo × grupo_edad × nivel)")
print("  · Medidas DAX     : recalcular tasas como SUM(numerador)/SUM(denominador), NUNCA promediar tasas")
print("  · Filtro de calidad: usar la columna 'confiable' (n_muestral >= 30)")

---
## 6. Cierre y siguientes pasos

**Estado del pipeline**

| Capa | Artefacto | Contenido |
|---|---|---|
| 🥉 Bronze | `data/bronze/externas/enemdu/` | CSV + XLSX originales (inmutables) |
| 🥈 Silver | `data/silver/enemdu/enemdu_unificado.parquet` | 2021-2025 unificado, limpio y decodificado |
| 🥇 Gold | `data/gold/enemdu/*.parquet` | 7 tablas agregadas + CSV para Power BI |
| 📄 Reportes | `reports/schemas/*.json` | Esquemas, calidad de Silver y manifiesto de Gold |

**Ejecución reproducible fuera del notebook**
```bash
python scripts/run_enemdu_pipeline.py --all
python scripts/run_enemdu_pipeline.py --silver --gold --years 2024 2025
```

**Siguientes pasos sugeridos**
1. **Validar los mapeos con `mapeo_educacion`/`mapeo_condact` en `fallback`**: si algún año cae en fallback, revisar el diccionario y ajustar `EDU_PATTERNS`/`CONDACT_PATTERNS` en `enemdu_mappings.py`.
2. **Refinar la sobrecalificación**: complementar el criterio normativo CIUO-08 con el criterio estadístico (modal por ocupación) y comparar resultados.
3. **Cruzar con datos internos** (graduados UCUENCA) para pasar de una línea base nacional a una comparación institucional.
4. **Errores estándar**: incorporar el diseño muestral complejo (conglomerado/estrato) si se van a publicar intervalos de confianza.